# OSU DEMO

This file expects to be in chipwhisperer/jupyter/courses/fault101. If you opened this in vscode, your local directory might be different. you can either cd' inside this notebook (not in the terminal) to where it expects to be, or adjust the other commands. Specifically, the compiling and programming steps below

In [ ]:
SCOPETYPE = 'OPENADC'
PLATFORM = 'CW308_NEORV32'
SS_VER = 'SS_VER_2_1'

In [ ]:
%run "./jupyter/Setup_Scripts/Setup_Generic.ipynb"

In [ ]:
%%bash -s "$PLATFORM" "$SS_VER"
cd ./firmware/mcu/simpleserial-glitch-tiny-aes
make PLATFORM=$1 CRYPTO_TARGET=NONE SS_VER=$2 -j OPT=0

In [ ]:
# Flash program, init comms.
fw_path = "./firmware/mcu/simpleserial-glitch-tiny-aes/simpleserial-glitch-tiny-aes-{}.bin".format(PLATFORM)
cw.program_target(scope, prog, fw_path)
if SS_VER == 'SS_VER_2_1':
    target.reset_comms()

In [ ]:
# Define reboot function. This is different for the ICE40.
def reboot_flush():
    #reset_target(scope) # <-- use this for other boards!
    # no reboot on the ICE40 since it doesn't have a way to reset it externally. We just need to reprogram it.
    cw.program_target(scope, prog, fw_path) 
    #Flush garbage too
    target.flush()

In [ ]:
# Set default glitch parameters.
scope.cglitch_setup()

In [ ]:
# Define define graph parameters
gc = cw.GlitchController(groups=["success", "reset", "normal"], parameters=["width", "offset", "ext_offset"])
gc.display_stats()

In [ ]:
# Define graph.
gc.glitch_plot(plotdots={"success":"+g", "reset":"xr", "normal":None}, x_index="ext_offset", y_index="offset")

In [ ]:
from tqdm.notebook import tqdm
import re
import struct
import time

# Number of runs test we test at each glitch configuration.
sample_size = 1

# Range of cycles we will search around target cycle.
ERROR_TOLERANCE = 20 

# Target cycle calculated from RTL simulation.
IF_CONDITION = 378180

# Cycles spent bootstrapping and initializing I/O. Chipwhisperer does not count that time. So, subtract that time
MAIN_START = 5352

# Chipwhisperer adds two instructions before the funciton call we are simulating. So, Add that time.
CYCLES_FOR_STORE = 6  # Cycles for a storing a word, i.e., 'sw'
CYCLES_FOR_ADD = 2    # Cycles to add an immediate, i.e., 'addi'
OFFSET = CYCLES_FOR_STORE + CYCLES_FOR_ADD - MAIN_START # Two extra instructions from CW - overhead. Should be negative.


# Note 1: Cycle count found running program and dividing the scope.adc.trig_count, after trigger has been lowered, by 4.
# Note 2: The ADC increments four times each cycle by default. If your numbers are very off, check!
PROGRAM_AVERAGE_CYCLE_COUNT = 494351             # Number of cycles a normal execution takes.
SIMULATION_MAX_CYCLE_COUNT = 378358 - MAIN_START # Number of cycles a simulated execution takes.
TIME_SCALAR = PROGRAM_AVERAGE_CYCLE_COUNT/SIMULATION_MAX_CYCLE_COUNT # Quotient of the two program runs.

# Just after test function returns
trigger_lower_cycle = 378248

# Offset start instruction and scale simulated result to match concreate results.
TARGET_CYCLE = PROGRAM_AVERAGE_CYCLE_COUNT - int(TIME_SCALAR*68) - 45 - 10# Time to lower trigger
    
# Set up glitch parameters ranges and step.
gc.set_range("width", 3500, 4500)        # Default values from the solution script
#gc.set_range("offset", 2000, 3200)       # Default values from the solution script
gc.set_range("offset", 2300, 2300)
gc.set_global_step([400, 200, 100])      # Default values from the solution script
#gc.set_range("width", 4300, 4500)       # custom values
#gc.set_range("offset", 2300, 2800)      # custom values
gc.set_range("ext_offset", TARGET_CYCLE - ERROR_TOLERANCE, TARGET_CYCLE + ERROR_TOLERANCE)
#gc.set_range("ext_offset", 40, 200)
#gc.set_range("ext_offset", 1, PROGRAM_MAX_CYCLE_COUNT)
gc.set_step("ext_offset", 1) # We are interested in the whole cycle search space; set step to 1.

scope.glitch.repeat = 1 # This says how many pulses the glitch signal has, not how many times we try. Check docs before touching it!
reboot_flush()
scope.adc.timeout = 1.5 # Number, in seconds, scope will wait before aborting a capture.

hitList = list()        # Holds time and parameters for successful runs
failList = list()       # Holds time and parameters for crashed runs
normalList = list()     # Holds time and parameters for benign runs
counter = 1             # Attempt counter.

clock_ID = time.CLOCK_MONOTONIC # Clock considers time since boot, including time the system has been suspended.
start_time = time.clock_gettime(clock_ID) # Get some time, in seconds. This will be our start time.
end_time = start_time # We will update this value each iteration.

for glitch_settings in gc.glitch_values():
    #start_time = time.time_ns()
    scope.glitch.offset = glitch_settings[1]
    scope.glitch.width = glitch_settings[0]
    scope.glitch.ext_offset = glitch_settings[2]
    for i in range(sample_size):
        if scope.adc.state:
            # can detect crash here (fast) before timing out (slow)
            failList.append((end_time - start_time, counter, scope.adc.trig_count/4, scope.glitch.width, scope.glitch.offset, scope.glitch.ext_offset))
            gc.add("reset")
            #Device is slow to boot?
            reboot_flush()

        cw.program_target(scope, prog, fw_path)
        scope.arm()
        data = bytearray([0]*5)
        target.simpleserial_write('p', data)
        #target.send_cmd("p", "P", bytearray([0]*5))
        ret = scope.capture()

        # Gather list elements; number of cycles + parameters. ADC counts 4 times each cycle, so we need to divide its count by 4.
        listElement = (end_time - start_time, counter, scope.adc.trig_count/4, scope.glitch.width, scope.glitch.offset, scope.glitch.ext_offset)
        
        if ret:
            listElement = (end_time - start_time, counter, scope.adc.trig_count/4, scope.glitch.width, scope.glitch.offset, scope.glitch.ext_offset)
            failList.append(listElement)
            gc.add("reset")
            
            #Device is slow to boot?
            reboot_flush()
        else:
            val = target.simpleserial_read_witherrors('r', 1, glitch_timeout=10, timeout=50) #For loop check
            if val['valid'] is False:
                listElement = (end_time - start_time, counter, scope.adc.trig_count/4, scope.glitch.width, scope.glitch.offset, scope.glitch.ext_offset)
                failList.append(listElement)
                gc.add("reset")
            else:

                if val['payload'] == bytearray([1]): #for loop check
                    listElement = (end_time - start_time, counter, scope.adc.trig_count/4, scope.glitch.width, scope.glitch.offset, scope.glitch.ext_offset)
                    hitList.append(listElement)
                    print(end_time-start_time)
                    gc.add("success")
                else:
                    listElement = (end_time - start_time, counter, scope.adc.trig_count/4, scope.glitch.width, scope.glitch.offset, scope.glitch.ext_offset)
                    normalList.append(listElement)
                    gc.add("normal")        
        end_time = time.clock_gettime(clock_ID) # Update timer
        counter = counter + 1                   # Update interation counter

In [ ]:
# Dump results into .txt files
with open('failList.txt', 'w') as file:
    for item in failList:
        file.write(str(item) + "," + '\n')
with open('hitList.txt', 'w') as file:
    for item in hitList:
        file.write(str(item) + "," + '\n')
with open('normalList.txt', 'w') as file:
    for item in normalList:
        file.write(str(item) + "," + '\n')

In [ ]:
hitList

In [ ]:
failList

In [ ]:
normalList #494370.0

In [ ]:
results = gc.calc(ignore_params=["width", "offset"], sort="success_rate")
results

And one for your width/offset settings:

In [ ]:
results = gc.calc(sort="total")
results

In [ ]:
scope.dis()
target.dis()

In [ ]:
assert broken is True

In [2]:
# Number of runs test we test at each glitch configuration.
sample_size = 1

# Range of cycles we will search around target cycle.
ERROR_TOLERANCE = 20 

# Target cycle calculated from RTL simulation.
IF_CONDITION = 378180

# Cycles spent bootstrapping and initializing I/O. Chipwhisperer does not count that time. So, subtract that time
MAIN_START = 5352

# Chipwhisperer adds two instructions before the funciton call we are simulating. So, Add that time.
CYCLES_FOR_STORE = 6  # Cycles for a storing a word, i.e., 'sw'
CYCLES_FOR_ADD = 2    # Cycles to add an immediate, i.e., 'addi'
OFFSET = CYCLES_FOR_STORE + CYCLES_FOR_ADD - MAIN_START # Two extra instructions from CW - overhead. Should be negative.


# Note 1: Cycle count found running program and dividing the scope.adc.trig_count, after trigger has been lowered, by 4.
# Note 2: The ADC increments four times each cycle by default. If your numbers are very off, check!
PROGRAM_AVERAGE_CYCLE_COUNT = 494351             # Number of cycles a normal execution takes.
SIMULATION_MAX_CYCLE_COUNT = 378358 - MAIN_START # Number of cycles a simulated execution takes.
SIMULATION_IF_CYCLE_COUNT = IF_CONDITION - MAIN_START
TIME_SCALAR = PROGRAM_AVERAGE_CYCLE_COUNT/SIMULATION_MAX_CYCLE_COUNT # Quotient of the two program runs.
INVERSE_SCALE = SIMULATION_MAX_CYCLE_COUNT/PROGRAM_AVERAGE_CYCLE_COUNT

time_to_lower_trigger = 55
extra_offset = 20
# Offset start instruction and scale simulated result to match concreate results.
TARGET_CYCLE = PROGRAM_AVERAGE_CYCLE_COUNT - int(TIME_SCALAR*68) - time_to_lower_trigger - extra_offset # This to seemed to be off a little, make sure to note it in the paper
print(int((SIMULATION_IF_CYCLE_COUNT) * TIME_SCALAR ))
print(int((SIMULATION_MAX_CYCLE_COUNT) * TIME_SCALAR - time_to_lower_trigger))
print(PROGRAM_AVERAGE_CYCLE_COUNT - int(TIME_SCALAR*68) - 55)
print(INVERSE_SCALE)

494115
494295
494206
0.7545367562723652
